# 🚀 Inference Engineering Benchmarks
## TensorRT-LLM (Local) vs NVIDIA NIM (Cloud API)

**Author:** Abishek Bangalore Muralikrishna  
**Purpose:** NVIDIA Hiring Portfolio — Inference Engineering Depth  
**Target:** Google Colab Free Tier (T4 GPU, 16GB VRAM)  
**Date:** May 2025  

---

### What This Notebook Demonstrates

| Skill | Evidence |
|-------|----------|
| TensorRT-LLM engine building & quantization | BF16/FP16, FP8, INT8-AWQ engines built from scratch |
| NVIDIA NIM cloud API integration | OpenAI-compatible endpoint with auth, streaming, metrics |
| Inference optimization analysis | TTFT, inter-token latency, throughput vs concurrency trade-offs |
| GPU capability awareness | Dynamic precision selection based on compute capability |
| Production-ready benchmarking | Statistical rigor (P50/P95/P99), concurrent request scaling |

---

## Cell 1: Environment Setup & GPU Detection

This cell:
- Detects GPU type and VRAM
- Checks compute capability (determines available quantization formats)
- Installs TensorRT-LLM and dependencies
- Validates the installation

In [ ]:
# ============================================================
# Cell 1: Environment Setup & GPU Detection
# ============================================================
import subprocess
import sys
import os

# Ensure PyTorch is available before GPU detection (bare VMs / fresh kernels)
try:
    import torch  # noqa: F401
except ImportError:
    print("Installing PyTorch...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch"],
        check=False,
        timeout=600,
    )
    import torch

# --- GPU Detection ---
print("=" * 60)
print("🔍 GPU Detection")
print("=" * 60)

gpu_name = "Unknown"
gpu_vram_mb = 0
compute_capability = (0, 0)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
        compute_capability = torch.cuda.get_device_capability(0)
        print(f"  GPU:              {gpu_name}")
        print(f"  VRAM:             {gpu_vram_mb:.0f} MB ({gpu_vram_mb/1024:.1f} GB)")
        print(f"  Compute Capability: {compute_capability[0]}.{compute_capability[1]}")
    else:
        print("  ⚠️  No GPU detected! This notebook requires a GPU runtime.")
        print("  Go to Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("  PyTorch not yet installed, will install below.")

# --- Compute Capability → Available Precisions ---
cc_major, cc_minor = compute_capability

SUPPORTS_FP8 = cc_major > 8 or (cc_major == 8 and cc_minor >= 9)  # Ada Lovelace+
SUPPORTS_BF16 = cc_major >= 8  # Ampere+
SUPPORTS_FP16 = True  # All CUDA GPUs
SUPPORTS_NVFP4 = cc_major >= 10  # Blackwell+

print(f"\n📊 Precision Support Matrix (based on SM {cc_major}.{cc_minor}):")
print(f"  FP16:     {'✅' if SUPPORTS_FP16 else '❌'}")
print(f"  BF16:     {'✅' if SUPPORTS_BF16 else '❌ (T4 is Turing — use FP16 instead)'}")
print(f"  FP8:      {'✅' if SUPPORTS_FP8 else '❌ (Requires Ada Lovelace / SM 8.9+)'}")
print(f"  NVFP4:    {'✅' if SUPPORTS_NVFP4 else '❌ (Requires Blackwell / SM 10.0+)'}")
print(f"  INT8-AWQ: ✅ (Weight-only quant, works on all GPUs)")

if not SUPPORTS_FP8:
    print(f"\n💡 Note: T4 (Turing) doesn't support FP8 natively. This notebook will:")
    print(f"   - Use FP16 as the baseline (analogous to BF16 on Ampere+)")
    print(f"   - Use INT8-AWQ as the primary optimization")
    print(f"   - Use the NVIDIA pre-quantized FP8 checkpoint (nvidia/Llama-3.1-8B-Instruct-FP8)")
    print(f"     with TRT-LLM auto-conversion (FP8 → runtime fallback)")
    print(f"   - Benchmark NIM Cloud API which runs on H100 (full FP8 support)")


In [ ]:
# --- Install Dependencies ---
print("=" * 60)
print("📦 Installing Dependencies")
print("=" * 60)

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "accelerate", "matplotlib",
    "pandas", "numpy", "requests"], timeout=300)

# Install TensorRT-LLM
# The pip wheel requires specific CUDA/PyTorch versions.
# We try the latest method first; fall back gracefully.
trt_llm_available = False

print("\n🔧 Attempting TensorRT-LLM installation...")
try:
    # First check if PyTorch CUDA is compatible
    import torch
    torch_version = torch.__version__
    cuda_version = torch.version.cuda
    print(f"  PyTorch: {torch_version}, CUDA: {cuda_version}")
    
    # Try installing tensorrt_llm
    # Pin torch version to prevent pip from downgrading
    with open("/tmp/torch-constraint.txt", "w") as f:
        f.write(f"torch=={torch_version}")
    
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "tensorrt_llm",
         "-c", "/tmp/torch-constraint.txt", "--no-deps"],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode != 0:
        print("  ⚠️  tensorrt_llm pip install failed. Trying full install...")
        result2 = subprocess.run(
            [sys.executable, "-m", "pip", "install", "tensorrt_llm"],
            capture_output=True, text=True, timeout=600
        )
        if result2.returncode != 0:
            print(f"  ❌ TensorRT-LLM installation failed.")
            print(f"  This is expected on some Colab CUDA versions.")
        else:
            trt_llm_available = True
    else:
        # Install remaining deps
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
            "mpi4py", "lru-dict", "setuptools", "wheel"], timeout=120)
        trt_llm_available = True

    if trt_llm_available:
        import tensorrt_llm
        print(f"  ✅ TensorRT-LLM {tensorrt_llm.__version__} installed successfully!")

except Exception as e:
    print(f"  ❌ TensorRT-LLM installation error: {e}")
    print(f"  Will use HuggingFace Transformers as local baseline instead.")

if not trt_llm_available:
    print("\n📋 Fallback Plan:")
    print("  Local benchmarks will use HuggingFace Transformers with:")
    print("  - FP16 (torch.cuda.amp)")
    print("  - Static quantization via bitsandbytes (INT8, INT4)")
    print("  - Flash Attention 2 (if available)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "bitsandbytes"], timeout=120, capture_output=True)
    # flash-attn requires build — skip silently if it fails
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "flash-attn", "--no-build-isolation"], timeout=300, capture_output=True)

print("\n✅ Setup complete!")

## Cell 2: Model Download & Configuration

We use **Meta Llama 3.1 8B Instruct** as our benchmark model. On T4 (16GB VRAM):

- **FP16**: ~16GB weights → tight fit, limited KV cache room
- **INT8-AWQ (W4A16)**: ~4-5GB weights → comfortable fit
- **Pre-quantized FP8 (NVIDIA Hub)**: ~8GB weights → good fit

We'll also download a smaller fallback model (Phi-3-mini) in case 8B doesn't fit.

In [ ]:
# ============================================================
# Cell 2: Model Download & Configuration
# ============================================================
import os
import json
from pathlib import Path

print("=" * 60)
print("📥 Model Configuration")
print("=" * 60)

# --- Model Selection ---
# Primary: Llama-3.1-8B-Instruct (the industry standard)
# Fallback: Phi-3-mini-4k-instruct (3.8B params, fits easily on T4)

MODEL_CONFIGS = {
    "llama-3.1-8b": {
        "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
        "nvidia_fp8_id": "nvidia/Llama-3.1-8B-Instruct-FP8",
        "nvidia_awq_id": "nvidia/Llama-3.1-8B-Instruct-INT8-AWQ",
        "nparams": "8B",
        "fp16_size_gb": 16.0,
        "awq_size_gb": 5.0,
        "fp8_size_gb": 8.0,
        "nim_model_id": "meta/llama-3.1-8b-instruct",
    },
    "phi-3-mini": {
        "hf_id": "microsoft/Phi-3-mini-4k-instruct",
        "nparams": "3.8B",
        "fp16_size_gb": 7.6,
        "nim_model_id": "microsoft/phi-4-mini-instruct",
    },
}

# Determine which model to use based on VRAM
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    vram_gb = 0
    print("⚠️  No CUDA device — select GPU runtime or expect CPU/OOM failures.")

# 8B in FP16 needs ~16GB + KV cache room, so use quantized or smaller model
if vram_gb >= 24:
    PRIMARY_MODEL = "llama-3.1-8b"
    DEFAULT_PRECISION = "fp16"
elif vram_gb >= 15:
    PRIMARY_MODEL = "llama-3.1-8b"
    DEFAULT_PRECISION = "awq"  # Must use quantized to fit
    print("💡 ~16GB class GPU — using quantized models to fit VRAM")
elif vram_gb > 0:
    PRIMARY_MODEL = "phi-3-mini"
    DEFAULT_PRECISION = "fp16"
    print("⚠️  Low VRAM — falling back to Phi-3-mini")
else:
    PRIMARY_MODEL = "phi-3-mini"
    DEFAULT_PRECISION = "fp16"
    print("⚠️  No GPU memory detected — using Phi-3-mini (still requires GPU for sensible benchmarks)")

model_cfg = MODEL_CONFIGS[PRIMARY_MODEL]
print(f"\n🎯 Selected Model: {model_cfg['hf_id']}")
print(f"   Parameters: {model_cfg['nparams']}")
print(f"   Default Precision: {DEFAULT_PRECISION}")
print(f"   NIM Cloud Model: {model_cfg['nim_model_id']}")

# --- HuggingFace Login (for gated models) ---
print("\n🔐 HuggingFace Authentication")
print("Llama 3.1 is a gated model. You need a HF token with access granted.")
print("Get one at: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct")

# Try multiple sources for the HF token (no interactive getpass — breaks Run All)
hf_token = ""
# 1. Environment variables
hf_token = os.environ.get("HF_TOKEN", os.environ.get("HUGGING_FACE_HUB_TOKEN", ""))
# 2. Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            print("  ✅ Loaded HF token from Colab secrets.")
    except Exception:
        pass
# 3. Interactive prompt (only if not running headless)
if not hf_token:
    # Auto-skip in Run All mode — set HF_TOKEN in Colab secrets to use Llama
    print("  ⚠️  No HF token found in env vars or Colab secrets.")
    print("  💡 To use Llama 3.1: Add HF_TOKEN to Colab secrets (🔑 in sidebar)")
    print("  📋 Falling back to Phi-3-mini (no auth required).")

if not hf_token and PRIMARY_MODEL == "llama-3.1-8b":
    print("⚠️  No HF token provided. Falling back to Phi-3-mini.")
    PRIMARY_MODEL = "phi-3-mini"
    DEFAULT_PRECISION = "fp16"
    model_cfg = MODEL_CONFIGS[PRIMARY_MODEL]
elif hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("✅ Logged in to HuggingFace Hub")

print(f"\n✅ Final model: {model_cfg['hf_id']} ({model_cfg['nparams']} params)")


In [ ]:
# --- Download Model Weights ---
print("=" * 60)
print("⬇️  Downloading Model Weights")
print("=" * 60)

from transformers import AutoTokenizer

model_id = model_cfg["hf_id"]
print(f"Loading tokenizer: {model_id}")

# Pass HF token if available (required for gated models like Llama)
tokenizer_kwargs = {"trust_remote_code": True}
if hf_token:
    tokenizer_kwargs["token"] = hf_token

tokenizer = AutoTokenizer.from_pretrained(model_id, **tokenizer_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded (vocab size: {len(tokenizer)})")

# We'll download model weights on-demand in each benchmark cell
# to manage memory carefully
print("\n📋 Model weights will be loaded on-demand per benchmark to conserve VRAM.")
print("   Each benchmark cell handles its own model loading/unloading.")

## Cell 3: Build TensorRT-LLM Engines

TensorRT-LLM compiles model graphs into optimized TensorRT engines. This is the **build step** that separates production inference from naive HuggingFace generation.

### Precision Strategy for T4 (SM 7.5):

| Precision | Method | Size | T4 Support | Notes |
|-----------|--------|------|------------|-------|
| FP16 | Native half | ~16GB | ✅ | Baseline (analogous to BF16 on Ampere+) |
| INT8-AWQ (W4A16) | Weight-only quant | ~5GB | ✅ | Best T4 optimization |
| FP8 | Post-training quant | ~8GB | ❌ Runtime | Build engine, note HW limitation |

> **Key insight:** Even though T4 can't run FP8 natively, building the FP8 engine
> demonstrates the workflow. On A100/H100, this would deliver 2x throughput over FP16.

In [ ]:
# ============================================================
# Cell 3: Build TensorRT-LLM Engines
# ============================================================
import time
import gc
import shutil

engine_build_results = []

if not trt_llm_available:
    print("⏭️  TensorRT-LLM not available. Skipping engine builds.")
    print("   Local benchmarks will use HuggingFace Transformers instead.")
else:
    from tensorrt_llm import LLM, SamplingParams
    from tensorrt_llm.llmapi import QuantConfig, QuantAlgo, CalibConfig
    
    # Define engine configurations to build
    ENGINE_CONFIGS = []
    
    # 1. FP16 (always available)
    if SUPPORTS_FP16:
        ENGINE_CONFIGS.append({
            "name": "FP16",
            "dtype": "float16",
            "quant_config": None,
            "calib_config": None,
        })
    
    # 2. BF16 (Ampere+)
    if SUPPORTS_BF16:
        ENGINE_CONFIGS.append({
            "name": "BF16",
            "dtype": "bfloat16",
            "quant_config": None,
            "calib_config": None,
        })
    
    # 3. FP8 (Ada Lovelace+) with calibration
    if SUPPORTS_FP8:
        ENGINE_CONFIGS.append({
            "name": "FP8",
            "dtype": "auto",
            "quant_config": QuantConfig(
                quant_algo=QuantAlgo.FP8,
                kv_cache_quant_algo=QuantAlgo.FP8,
            ),
            "calib_config": CalibConfig(
                calib_dataset="cnn_dailymail",
                calib_batches=128,  # Reduced for Colab speed
                calib_max_seq_length=256,
            ),
        })
    
    # 4. INT8-AWQ (weight-only, works on all GPUs)
    ENGINE_CONFIGS.append({
        "name": "INT8-AWQ",
        "dtype": "float16",
        "quant_config": QuantConfig(quant_algo=QuantAlgo.W4A16_AWQ),
        "calib_config": None,
    })
    
    # 5. Pre-quantized FP8 from NVIDIA Hub (if available for this model)
    if "nvidia_fp8_id" in model_cfg and SUPPORTS_FP8:
        ENGINE_CONFIGS.append({
            "name": "FP8-NVIDIA-Hub",
            "model_override": model_cfg["nvidia_fp8_id"],
            "dtype": "auto",
            "quant_config": None,  # Already quantized
            "calib_config": None,
        })
    
    print(f"🔧 Building {len(ENGINE_CONFIGS)} TensorRT-LLM engines...")
    print(f"   Model: {model_id}")
    print()
    
    for cfg in ENGINE_CONFIGS:
        engine_name = cfg["name"]
        model_to_use = cfg.get("model_override", model_id)
        print(f"\n{'─' * 50}")
        print(f"🔨 Building: {engine_name}")
        print(f"   Model: {model_to_use}")
        print(f"   dtype: {cfg.get('dtype', 'auto')}")
        
        build_kwargs = {
            "model": model_to_use,
            "dtype": cfg.get("dtype", "auto"),
            "trust_remote_code": True,
            "max_batch_size": 16,
            "max_input_len": 2048,
            "max_output_len": 512,
        }
        if cfg.get("quant_config"):
            build_kwargs["quant_config"] = cfg["quant_config"]
        if cfg.get("calib_config"):
            build_kwargs["calib_config"] = cfg["calib_config"]
        
        try:
            t0 = time.time()
            llm = LLM(**build_kwargs)
            build_time = time.time() - t0
            
            # Get engine size
            engine_dir = Path(".") / ".trt_llm_engine"
            engine_size_mb = 0
            if engine_dir.exists():
                engine_size_mb = sum(
                    f.stat().st_size for f in engine_dir.rglob("*") if f.is_file()
                ) / (1024**2)
            
            # Quick sanity check
            sampling_params = SamplingParams(max_tokens=10, temperature=0.0)
            output = llm.generate(["Hello, world!"], sampling_params)
            sanity_text = output[0].outputs[0].text[:50]
            
            result = {
                "engine": engine_name,
                "model": model_to_use,
                "build_time_s": round(build_time, 1),
                "engine_size_mb": round(engine_size_mb, 1),
                "status": "✅ Success",
                "sanity_output": sanity_text,
                "llm": llm,  # Keep reference for benchmarking
            }
            engine_build_results.append(result)
            
            print(f"   ✅ Build time: {build_time:.1f}s")
            print(f"   Engine size: {engine_size_mb:.1f} MB")
            print(f"   Sanity check: '{sanity_text}'")
            
        except torch.cuda.OutOfMemoryError:
            print(f"   ❌ OOM — model too large for this precision on T4")
            engine_build_results.append({
                "engine": engine_name,
                "status": "❌ OOM",
                "build_time_s": None,
                "engine_size_mb": None,
            })
            gc.collect()
            torch.cuda.empty_cache()
        except RuntimeError as e:
            if "FP8" in str(e) or "Unsupported" in str(e):
                print(f"   ❌ FP8 not supported on this GPU (SM {cc_major}.{cc_minor})")
                print(f"      This is expected on T4. FP8 requires Ada Lovelace (SM 8.9+).")
            else:
                print(f"   ❌ Runtime error: {e}")
            engine_build_results.append({
                "engine": engine_name,
                "status": f"❌ {str(e)[:60]}",
                "build_time_s": None,
                "engine_size_mb": None,
            })
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"   ❌ Error: {type(e).__name__}: {e}")
            engine_build_results.append({
                "engine": engine_name,
                "status": f"❌ {type(e).__name__}",
                "build_time_s": None,
                "engine_size_mb": None,
            })
            gc.collect()
            torch.cuda.empty_cache()

    # Print build summary
    print(f"\n{'=' * 60}")
    print("📊 Engine Build Summary")
    print(f"{'=' * 60}")
    for r in engine_build_results:
        status = r['status']
        name = r['engine']
        bt = f"{r['build_time_s']:.1f}s" if r.get('build_time_s') else 'N/A'
        sz = f"{r['engine_size_mb']:.0f} MB" if r.get('engine_size_mb') else 'N/A'
        print(f"  {name:20s} | {status:20s} | Build: {bt:10s} | Size: {sz}")

## Cell 4: Local Benchmarks (TensorRT-LLM / HuggingFace Transformers)

### Metrics Collected

| Metric | Description | Why It Matters |
|--------|-------------|----------------|
| **TTFT** | Time to First Token | User-perceived latency (prefill speed) |
| **ITL** | Inter-Token Latency | Generation smoothness |
| **Tokens/sec** | Output throughput | Raw decode speed |
| **Throughput** | Requests/sec at concurrency N | Serving capacity |

### Benchmark Dimensions
- **Input lengths**: 128, 512, 1024, 2048 tokens
- **Concurrency**: 1, 4, 8, 16 concurrent requests
- **Output length**: 128 tokens (fixed for fairness)

In [ ]:
# ============================================================
# Cell 4: Local Benchmarks
# ============================================================
import numpy as np
import pandas as pd
import time
import gc
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from typing import List, Optional

# --- Benchmark Configuration ---
INPUT_LENGTHS = [128, 512, 1024, 2048]
CONCURRENCY_LEVELS = [1, 4, 8, 16]
OUTPUT_TOKENS = 128
WARMUP_RUNS = 2
BENCHMARK_RUNS = 5  # Per configuration

@dataclass
class BenchmarkResult:
    """Single benchmark measurement."""
    engine: str
    precision: str
    input_length: int
    concurrency: int
    output_tokens: int
    ttft_ms: float          # Time to first token
    itl_ms: float           # Inter-token latency (mean)
    tokens_per_sec: float   # Decode throughput
    total_latency_ms: float # End-to-end
    gpu_mem_mb: float       # Peak GPU memory
    backend: str            # 'trt_llm' or 'hf_transformers'

all_local_results: List[BenchmarkResult] = []

def generate_prompt(tokenizer, target_length: int) -> str:
    """Generate a prompt of approximately target_length tokens."""
    # Use a base paragraph and repeat/pad to target length
    base_text = (
        "Artificial intelligence has transformed numerous industries, from healthcare "
        "to finance, enabling unprecedented levels of automation and insight. "
        "Machine learning models, particularly large language models, have demonstrated "
        "remarkable capabilities in natural language understanding, generation, and reasoning. "
        "The development of efficient inference systems is crucial for deploying these models "
        "at scale, balancing latency, throughput, and cost considerations. "
    )
    tokens = tokenizer.encode(base_text)
    while len(tokens) < target_length:
        tokens = tokens + tokens[:max(1, target_length - len(tokens))]
    return tokenizer.decode(tokens[:target_length])

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB."""
    import torch
    return torch.cuda.memory_allocated() / (1024**2)

print("✅ Benchmark utilities loaded.")

In [ ]:
# --- TensorRT-LLM Benchmarks ---

if trt_llm_available and any(r.get('llm') for r in engine_build_results):
    from tensorrt_llm import SamplingParams
    
    successful_engines = [r for r in engine_build_results if r.get('llm')]
    
    for engine_result in successful_engines:
        engine_name = engine_result['engine']
        llm = engine_result['llm']
        
        print(f"\n{'=' * 60}")
        print(f"🏎️  Benchmarking: {engine_name}")
        print(f"{'=' * 60}")
        
        # --- Latency Benchmark (single request, varying input lengths) ---
        print(f"\n  📏 Single-Request Latency")
        
        for input_len in INPUT_LENGTHS:
            prompt = generate_prompt(tokenizer, input_len)
            actual_len = len(tokenizer.encode(prompt))
            
            # Warmup
            for _ in range(WARMUP_RUNS):
                _ = llm.generate([prompt], SamplingParams(max_tokens=10, temperature=0.0))
            
            torch.cuda.synchronize()
            mem_before = get_gpu_memory_mb()
            
            # Timed runs
            ttft_list = []
            total_lat_list = []
            output_tok_list = []
            
            for _ in range(BENCHMARK_RUNS):
                torch.cuda.synchronize()
                t_start = time.perf_counter()
                
                outputs = llm.generate(
                    [prompt],
                    SamplingParams(max_tokens=OUTPUT_TOKENS, temperature=0.0)
                )
                
                torch.cuda.synchronize()
                t_end = time.perf_counter()
                
                total_latency = (t_end - t_start) * 1000  # ms
                n_output_tokens = len(outputs[0].outputs[0].token_ids)
                
                # Estimate TTFT (prefill time ≈ total - decode time)
                # TRT-LLM provides timing info if available
                if hasattr(outputs[0], 'prefill_time') and outputs[0].prefill_time:
                    ttft = outputs[0].prefill_time * 1000
                else:
                    # Approximate: assume decode is linear, estimate TTFT
                    decode_time_est = (n_output_tokens / OUTPUT_TOKENS) * total_latency * 0.85
                    ttft = total_latency - decode_time_est
                
                ttft_list.append(ttft)
                total_lat_list.append(total_latency)
                output_tok_list.append(n_output_tokens)
            
            mem_after = get_gpu_memory_mb()
            
            avg_ttft = np.mean(ttft_list)
            avg_total = np.mean(total_lat_list)
            avg_output = np.mean(output_tok_list)
            tokens_per_sec = (avg_output / avg_total) * 1000
            itl = (avg_total - avg_ttft) / max(avg_output - 1, 1)  # ms per token after first
            
            result = BenchmarkResult(
                engine=engine_name,
                precision=engine_name,
                input_length=actual_len,
                concurrency=1,
                output_tokens=int(avg_output),
                ttft_ms=round(avg_ttft, 2),
                itl_ms=round(itl, 2),
                tokens_per_sec=round(tokens_per_sec, 2),
                total_latency_ms=round(avg_total, 2),
                gpu_mem_mb=round(mem_after, 1),
                backend='trt_llm',
            )
            all_local_results.append(result)
            
            print(f"    Input {actual_len:5d} tok | TTFT: {avg_ttft:7.1f}ms | "
                  f"ITL: {itl:5.1f}ms | T/s: {tokens_per_sec:6.1f} | "
                  f"Mem: {mem_after:.0f}MB")
        
        # --- Throughput Benchmark (varying concurrency) ---
        print(f"\n  📈 Concurrent Request Throughput")
        
        prompt_512 = generate_prompt(tokenizer, 512)
        
        for concurrency in CONCURRENCY_LEVELS:
            prompts = [prompt_512] * concurrency
            
            # Warmup
            _ = llm.generate(prompts[:1], SamplingParams(max_tokens=10, temperature=0.0))
            
            torch.cuda.synchronize()
            
            # Timed run
            t_start = time.perf_counter()
            outputs = llm.generate(
                prompts,
                SamplingParams(max_tokens=OUTPUT_TOKENS, temperature=0.0)
            )
            torch.cuda.synchronize()
            t_end = time.perf_counter()
            
            total_time_s = t_end - t_start
            total_output_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
            total_input_tokens = sum(len(tokenizer.encode(p)) for p in prompts)
            throughput = concurrency / total_time_s  # requests/sec
            output_tps = total_output_tokens / total_time_s  # output tokens/sec
            
            result = BenchmarkResult(
                engine=engine_name,
                precision=engine_name,
                input_length=512,
                concurrency=concurrency,
                output_tokens=int(np.mean([len(o.outputs[0].token_ids) for o in outputs])),
                ttft_ms=round(total_time_s * 1000 / concurrency, 2),  # Avg per request
                itl_ms=round((total_time_s * 1000) / total_output_tokens, 2),
                tokens_per_sec=round(output_tps, 2),
                total_latency_ms=round(total_time_s * 1000, 2),
                gpu_mem_mb=round(get_gpu_memory_mb(), 1),
                backend='trt_llm',
            )
            all_local_results.append(result)
            
            print(f"    Concurrency {concurrency:2d} | Total: {total_time_s*1000:7.0f}ms | "
                  f"Req/s: {throughput:5.1f} | Out T/s: {output_tps:7.1f}")
        
        # Clean up this engine to free VRAM for next one
        if hasattr(llm, 'shutdown'):
            llm.shutdown()
        del llm
        gc.collect()
        torch.cuda.empty_cache()
        print(f"\n  🧹 Engine {engine_name} released.")

else:
    print("⏭️  No TRT-LLM engines available. Will use HuggingFace Transformers fallback.")

In [ ]:
# --- HuggingFace Transformers Fallback Benchmarks ---
# Used when TRT-LLM is not available, or as additional baseline

import torch
from transformers import AutoModelForCausalLM, pipeline

HF_BENCHMARK_CONFIGS = []

# Always add FP16 baseline
HF_BENCHMARK_CONFIGS.append({
    "name": "HF-FP16",
    "torch_dtype": torch.float16,
    "load_in_8bit": False,
    "load_in_4bit": False,
})

# Add INT8 via bitsandbytes if available
try:
    import bitsandbytes
    HF_BENCHMARK_CONFIGS.append({
        "name": "HF-INT8",
        "torch_dtype": torch.float16,
        "load_in_8bit": True,
        "load_in_4bit": False,
    })
    HF_BENCHMARK_CONFIGS.append({
        "name": "HF-INT4-AWQ",
        "torch_dtype": torch.float16,
        "load_in_8bit": False,
        "load_in_4bit": True,
    })
except ImportError:
    print("⚠️  bitsandbytes not available. INT8/INT4 benchmarks skipped.")

if not trt_llm_available or len([r for r in all_local_results if r.backend == 'trt_llm']) == 0:
    print(f"\n{'=' * 60}")
    print(f"🏎️  HuggingFace Transformers Benchmarks (Fallback)")
    print(f"{'=' * 60}")
    
    for cfg in HF_BENCHMARK_CONFIGS:
        engine_name = cfg["name"]
        print(f"\n  Loading: {engine_name}")
        
        try:
            # Free memory first
            gc.collect()
            torch.cuda.empty_cache()
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=cfg["torch_dtype"],
                device_map="auto",
                load_in_8bit=cfg["load_in_8bit"],
                load_in_4bit=cfg["load_in_4bit"],
                trust_remote_code=True,
                token=hf_token if hf_token else None,
            )
            model.eval()
            
            print(f"  ✅ Model loaded. GPU memory: {get_gpu_memory_mb():.0f} MB")
            
            # --- Single-Request Latency ---
            print(f"  📏 Single-Request Latency")
            
            for input_len in INPUT_LENGTHS:
                prompt = generate_prompt(tokenizer, input_len)
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                actual_len = inputs["input_ids"].shape[1]
                
                # Warmup
                for _ in range(WARMUP_RUNS):
                    with torch.no_grad():
                        _ = model.generate(**inputs, max_new_tokens=10, do_sample=False)
                
                torch.cuda.synchronize()
                
                # Timed runs
                ttft_list = []
                total_list = []
                out_tok_list = []
                
                for _ in range(BENCHMARK_RUNS):
                    torch.cuda.synchronize()
                    t_start = time.perf_counter()
                    
                    with torch.no_grad():
                        output_ids = model.generate(
                            **inputs,
                            max_new_tokens=OUTPUT_TOKENS,
                            do_sample=False,
                            use_cache=True,
                        )
                    
                    torch.cuda.synchronize()
                    t_end = time.perf_counter()
                    
                    total_ms = (t_end - t_start) * 1000
                    n_out = output_ids.shape[1] - actual_len
                    
                    total_list.append(total_ms)
                    out_tok_list.append(n_out)
                    
                    # Estimate TTFT: time for prefill vs decode
                    # Rough heuristic: prefill ≈ 30-50% for short, less for long
                    est_ttft = total_ms * (actual_len / (actual_len + OUTPUT_TOKENS)) * 0.5
                    ttft_list.append(est_ttft)
                
                avg_ttft = np.mean(ttft_list)
                avg_total = np.mean(total_list)
                avg_out = np.mean(out_tok_list)
                tps = (avg_out / avg_total) * 1000
                itl = (avg_total - avg_ttft) / max(avg_out - 1, 1)
                
                result = BenchmarkResult(
                    engine=engine_name,
                    precision=engine_name,
                    input_length=actual_len,
                    concurrency=1,
                    output_tokens=int(avg_out),
                    ttft_ms=round(avg_ttft, 2),
                    itl_ms=round(itl, 2),
                    tokens_per_sec=round(tps, 2),
                    total_latency_ms=round(avg_total, 2),
                    gpu_mem_mb=round(get_gpu_memory_mb(), 1),
                    backend='hf_transformers',
                )
                all_local_results.append(result)
                
                print(f"    Input {actual_len:5d} tok | TTFT: {avg_ttft:7.1f}ms | "
                      f"ITL: {itl:5.1f}ms | T/s: {tps:6.1f} | "
                      f"Mem: {get_gpu_memory_mb():.0f}MB")
            
            # Clean up
            del model
            gc.collect()
            torch.cuda.empty_cache()
            
        except torch.cuda.OutOfMemoryError:
            print(f"  ❌ OOM for {engine_name}")
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"  ❌ Error: {e}")
            gc.collect()
            torch.cuda.empty_cache()

# --- Display Local Results ---
if all_local_results:
    df_local = pd.DataFrame([asdict(r) for r in all_local_results])
    print(f"\n{'=' * 60}")
    print("📊 Local Benchmark Results")
    print(f"{'=' * 60}")
    display(df_local)
else:
    print("\n⚠️  No local benchmark results collected.")
    df_local = pd.DataFrame()

## Cell 5: NVIDIA NIM Cloud API Benchmarks

NVIDIA NIM (NeMo Inference Microservice) provides **cloud-hosted, production-optimized inference** via an OpenAI-compatible API. This is what you'd use in production when you don't want to manage GPU infrastructure.

### Key Points
- **Endpoint:** `https://integrate.api.nvidia.com/v1/chat/completions`
- **Auth:** Bearer token (NVIDIA Build API key)
- **Models:** Run on NVIDIA's infrastructure (H100/A100 clusters)
- **Benefits:** Full FP8 support, tensor parallelism, continuous batching

Get your API key at: https://build.nvidia.com/

In [ ]:
# ============================================================
# Cell 5: NVIDIA NIM Cloud API Benchmarks
# ============================================================
import os
import requests
import json
from getpass import getpass
from typing import List, Dict
import statistics
import time
import numpy as np

# --- API Key ---
print("🔑 NVIDIA Build API Key")
print("Get your key at: https://build.nvidia.com/ → Select a model → 'Get API Key'")
print()

# Try loading from Colab secrets first, then env vars
NIM_API_KEY = ""
try:
    from google.colab import userdata
    NIM_API_KEY = userdata.get('NVIDIA_BUILD_API_KEY') or ""
    if NIM_API_KEY:
        print("  ✅ Loaded from Colab secrets.")
except Exception:
    pass

if not NIM_API_KEY:
    NIM_API_KEY = os.environ.get("NVIDIA_BUILD_API_KEY", "")

if not NIM_API_KEY:
    # Interactive prompt — only works when running cell manually
    print("  💡 Add NVIDIA_BUILD_API_KEY to Colab secrets (🔑 in sidebar) for auto-load.")
    print("  Or set it as an environment variable.")
    try:
        NIM_API_KEY = getpass("  Enter your NVIDIA Build API key (or press Enter to skip): ")
    except Exception:
        print("  ⚠️  Cannot prompt for API key. NIM benchmarks will be skipped.")

if NIM_API_KEY:
    print("  ✅ API key set.")
else:
    print("  ⚠️  No API key provided. NIM benchmarks will be skipped.")

NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"
NIM_MODELS = [model_cfg["nim_model_id"]]

# Also test 70B if available
if PRIMARY_MODEL == "llama-3.1-8b":
    NIM_MODELS.append("meta/llama-3.1-70b-instruct")


In [ ]:
# --- NIM API Client ---
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

@dataclass
class NIMBenchmarkResult:
    """NIM cloud API benchmark measurement."""
    model: str
    input_length: int
    concurrency: int
    output_tokens: int
    ttft_ms: float
    itl_ms: float
    tokens_per_sec: float
    total_latency_ms: float
    p50_latency_ms: float
    p95_latency_ms: float
    p99_latency_ms: float
    backend: str = 'nim_cloud'

all_nim_results: List[NIMBenchmarkResult] = []

def call_nim_api(
    api_key: str,
    model: str,
    messages: List[Dict],
    max_tokens: int = 128,
    temperature: float = 0.0,
    stream: bool = False,
) -> dict:
    """Make a single request to the NIM API."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": stream,
    }
    
    resp = requests.post(
        f"{NIM_BASE_URL}/chat/completions",
        headers=headers,
        json=payload,
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

def call_nim_streaming_ttft(
    api_key: str,
    model: str,
    messages: List[Dict],
    max_tokens: int = 128,
    temperature: float = 0.0,
) -> dict:
    """Make a streaming request to measure TTFT precisely."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": True,
    }
    
    ttft = None
    first_token_time = None
    token_times = []
    n_tokens = 0
    
    t_start = time.perf_counter()
    
    with requests.post(
        f"{NIM_BASE_URL}/chat/completions",
        headers=headers,
        json=payload,
        timeout=120,
        stream=True,
    ) as resp:
        resp.raise_for_status()
        
        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue
            data_str = line[6:]  # Remove "data: "
            if data_str.strip() == "[DONE]":
                break
            
            try:
                data = json.loads(data_str)
                delta = data.get("choices", [{}])[0].get("delta", {})
                if "content" in delta and delta["content"]:
                    n_tokens += 1
                    token_time = time.perf_counter()
                    if ttft is None:
                        ttft = (token_time - t_start) * 1000
                    token_times.append(token_time)
            except json.JSONDecodeError:
                continue
    
    total_latency = (time.perf_counter() - t_start) * 1000
    
    return {
        "ttft_ms": ttft or 0,
        "total_latency_ms": total_latency,
        "n_output_tokens": n_tokens,
    }

print("✅ NIM API client ready.")

In [ ]:
# --- Run NIM Benchmarks ---

if NIM_API_KEY:
    for nim_model in NIM_MODELS:
        print(f"\n{'=' * 60}")
        print(f"☁️  NIM Cloud Benchmark: {nim_model}")
        print(f"{'=' * 60}")
        
        # Test API connectivity first
        try:
            test_resp = call_nim_api(
                NIM_API_KEY, nim_model,
                [{"role": "user", "content": "Hello!"}],
                max_tokens=5,
            )
            print(f"  ✅ API connected. Test response received.")
        except requests.exceptions.HTTPError as e:
            print(f"  ❌ API error: {e}")
            if e.response is not None and e.response.status_code == 401:
                print("  Check your API key.")
            continue
        except Exception as e:
            print(f"  ❌ Connection error: {e}")
            continue
        
        # --- Latency Benchmark (varying input lengths) ---
        print(f"\n  📏 Single-Request Latency (Streaming for TTFT)")
        
        for input_len in INPUT_LENGTHS:
            prompt = generate_prompt(tokenizer, input_len)
            messages = [{"role": "user", "content": prompt}]
            
            latencies = []
            ttfts = []
            output_toks = []
            
            for run_idx in range(BENCHMARK_RUNS):
                try:
                    result = call_nim_streaming_ttft(
                        NIM_API_KEY, nim_model, messages,
                        max_tokens=OUTPUT_TOKENS,
                    )
                    latencies.append(result["total_latency_ms"])
                    ttfts.append(result["ttft_ms"])
                    output_toks.append(result["n_output_tokens"])
                except Exception as e:
                    print(f"    ⚠️  Run {run_idx} failed: {e}")
                
                # Small delay to avoid rate limits
                time.sleep(0.5)
            
            if latencies:
                avg_ttft = np.mean(ttfts)
                avg_total = np.mean(latencies)
                avg_out = np.mean(output_toks)
                tps = (avg_out / avg_total) * 1000
                itl = (avg_total - avg_ttft) / max(avg_out - 1, 1)
                
                nim_result = NIMBenchmarkResult(
                    model=nim_model,
                    input_length=input_len,
                    concurrency=1,
                    output_tokens=int(avg_out),
                    ttft_ms=round(avg_ttft, 2),
                    itl_ms=round(itl, 2),
                    tokens_per_sec=round(tps, 2),
                    total_latency_ms=round(avg_total, 2),
                    p50_latency_ms=round(np.percentile(latencies, 50), 2),
                    p95_latency_ms=round(np.percentile(latencies, 95), 2),
                    p99_latency_ms=round(np.percentile(latencies, 99), 2),
                )
                all_nim_results.append(nim_result)
                
                print(f"    Input {input_len:5d} tok | TTFT: {avg_ttft:7.1f}ms | "
                      f"ITL: {itl:5.1f}ms | T/s: {tps:6.1f} | "
                      f"P95: {np.percentile(latencies, 95):.0f}ms")
        
        # --- Concurrent Request Throughput ---
        print(f"\n  📈 Concurrent Request Throughput")
        
        for concurrency in CONCURRENCY_LEVELS:
            prompt_512 = generate_prompt(tokenizer, 512)
            messages = [{"role": "user", "content": prompt_512}]
            
            latencies = []
            ttfts = []
            
            def _single_request(idx):
                """Single NIM request for concurrent execution."""
                try:
                    result = call_nim_streaming_ttft(
                        NIM_API_KEY, nim_model, messages,
                        max_tokens=OUTPUT_TOKENS,
                    )
                    return result
                except Exception as e:
                    return {"total_latency_ms": -1, "ttft_ms": -1, "n_output_tokens": 0, "error": str(e)}
            
            with ThreadPoolExecutor(max_workers=concurrency) as executor:
                futures = [executor.submit(_single_request, i) for i in range(concurrency)]
                results_list = [f.result() for f in as_completed(futures)]
            
            valid = [r for r in results_list if r.get("total_latency_ms", -1) > 0]
            if valid:
                total_output = sum(r["n_output_tokens"] for r in valid)
                max_latency = max(r["total_latency_ms"] for r in valid)
                throughput = len(valid) / (max_latency / 1000)
                avg_ttft = np.mean([r["ttft_ms"] for r in valid])
                output_tps = total_output / (max_latency / 1000)
                itl = (max_latency - avg_ttft) / max(total_output - 1, 1)
                
                nim_result = NIMBenchmarkResult(
                    model=nim_model,
                    input_length=512,
                    concurrency=concurrency,
                    output_tokens=int(np.mean([r["n_output_tokens"] for r in valid])),
                    ttft_ms=round(avg_ttft, 2),
                    itl_ms=round(itl, 2),
                    tokens_per_sec=round(output_tps, 2),
                    total_latency_ms=round(max_latency, 2),
                    p50_latency_ms=round(np.percentile([r["total_latency_ms"] for r in valid], 50), 2),
                    p95_latency_ms=round(np.percentile([r["total_latency_ms"] for r in valid], 95), 2),
                    p99_latency_ms=round(np.percentile([r["total_latency_ms"] for r in valid], 99), 2),
                )
                all_nim_results.append(nim_result)
                
                print(f"    Concurrency {concurrency:2d} | Max latency: {max_latency:7.0f}ms | "
                      f"Req/s: {throughput:5.1f} | Out T/s: {output_tps:7.1f}")
            
            time.sleep(1)  # Rate limit buffer
    
    # Display NIM results
    if all_nim_results:
        df_nim = pd.DataFrame([asdict(r) for r in all_nim_results])
        print(f"\n{'=' * 60}")
        print("📊 NIM Cloud Benchmark Results")
        print(f"{'=' * 60}")
        display(df_nim)
    else:
        print("\n⚠️  No NIM results collected.")
        df_nim = pd.DataFrame()
else:
    print("⏭️  No NIM API key. Skipping cloud benchmarks.")
    df_nim = pd.DataFrame()

## Cell 6: Comparison & Visualization

This is where it all comes together. We compare:

1. **Local TRT-LLM** (or HF Transformers) at various precisions
2. **NIM Cloud API** (running on H100 infrastructure)

### Key Questions
- How much does quantization improve throughput on T4?
- How does local T4 compare to cloud H100 for latency?
- What's the throughput scaling curve with concurrency?
- Where is the crossover point (latency vs throughput trade-off)?

In [ ]:
# ============================================================
# Cell 6: Comparison & Visualization
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# --- Merge Results ---
# Normalize all results into a single DataFrame for comparison
comparison_rows = []

# Local results
for r in all_local_results:
    comparison_rows.append({
        "source": f"Local: {r.engine}",
        "backend": r.backend,
        "precision": r.precision,
        "model": model_id,
        "input_length": r.input_length,
        "concurrency": r.concurrency,
        "output_tokens": r.output_tokens,
        "ttft_ms": r.ttft_ms,
        "itl_ms": r.itl_ms,
        "tokens_per_sec": r.tokens_per_sec,
        "total_latency_ms": r.total_latency_ms,
        "gpu_mem_mb": r.gpu_mem_mb,
        "p95_latency_ms": None,
    })

# NIM results
for r in all_nim_results:
    comparison_rows.append({
        "source": f"NIM: {r.model}",
        "backend": "nim_cloud",
        "precision": "FP8 (H100)",
        "model": r.model,
        "input_length": r.input_length,
        "concurrency": r.concurrency,
        "output_tokens": r.output_tokens,
        "ttft_ms": r.ttft_ms,
        "itl_ms": r.itl_ms,
        "tokens_per_sec": r.tokens_per_sec,
        "total_latency_ms": r.total_latency_ms,
        "gpu_mem_mb": None,
        "p95_latency_ms": r.p95_latency_ms,
    })

df_compare = pd.DataFrame(comparison_rows)

if df_compare.empty:
    print("⚠️  No benchmark results to visualize. Run the benchmark cells first.")
else:
    print(f"✅ Merged {len(df_compare)} benchmark results into comparison table.")

In [ ]:
# --- Visualization 1: TTFT vs Input Length ---

if not df_compare.empty:
    df_single = df_compare[df_compare["concurrency"] == 1].copy()
    
    if not df_single.empty:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # TTFT vs Input Length
        ax = axes[0]
        for source, group in df_single.groupby("source"):
            group_sorted = group.sort_values("input_length")
            ax.plot(group_sorted["input_length"], group_sorted["ttft_ms"],
                    marker='o', label=source, linewidth=2, markersize=6)
        ax.set_xlabel("Input Length (tokens)", fontsize=12)
        ax.set_ylabel("TTFT (ms)", fontsize=12)
        ax.set_title("Time to First Token vs Input Length", fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, loc='upper left')
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log', base=2)
        
        # Tokens/sec vs Input Length
        ax = axes[1]
        for source, group in df_single.groupby("source"):
            group_sorted = group.sort_values("input_length")
            ax.plot(group_sorted["input_length"], group_sorted["tokens_per_sec"],
                    marker='s', label=source, linewidth=2, markersize=6)
        ax.set_xlabel("Input Length (tokens)", fontsize=12)
        ax.set_ylabel("Tokens/sec", fontsize=12)
        ax.set_title("Decode Throughput vs Input Length", fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log', base=2)
        
        plt.tight_layout()
        plt.savefig("benchmark_ttft_throughput.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Saved: benchmark_ttft_throughput.png")

In [ ]:
# --- Visualization 2: Throughput vs Concurrency ---

if not df_compare.empty:
    df_conc = df_compare[df_compare["input_length"] == 512].copy()
    
    if not df_conc.empty:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Tokens/sec vs Concurrency
        ax = axes[0]
        for source, group in df_conc.groupby("source"):
            group_sorted = group.sort_values("concurrency")
            ax.plot(group_sorted["concurrency"], group_sorted["tokens_per_sec"],
                    marker='D', label=source, linewidth=2, markersize=6)
        ax.set_xlabel("Concurrent Requests", fontsize=12)
        ax.set_ylabel("Output Tokens/sec", fontsize=12)
        ax.set_title("Throughput vs Concurrency (512-token input)", fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Latency vs Concurrency
        ax = axes[1]
        for source, group in df_conc.groupby("source"):
            group_sorted = group.sort_values("concurrency")
            ax.plot(group_sorted["concurrency"], group_sorted["total_latency_ms"],
                    marker='v', label=source, linewidth=2, markersize=6)
        ax.set_xlabel("Concurrent Requests", fontsize=12)
        ax.set_ylabel("Total Latency (ms)", fontsize=12)
        ax.set_title("Latency vs Concurrency (512-token input)", fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig("benchmark_concurrency.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Saved: benchmark_concurrency.png")

In [ ]:
# --- Visualization 3: Latency Comparison Bar Chart ---

if not df_compare.empty:
    df_bar = df_compare[(df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)].copy()
    
    if not df_bar.empty:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        sources = df_bar["source"].unique()
        colors = plt.cm.Set2(np.linspace(0, 1, len(sources)))
        color_map = dict(zip(sources, colors))
        
        metrics = [
            ("ttft_ms", "Time to First Token (ms)", "TTFT Comparison"),
            ("itl_ms", "Inter-Token Latency (ms)", "ITL Comparison"),
            ("tokens_per_sec", "Tokens/sec", "Decode Throughput"),
        ]
        
        for ax, (col, ylabel, title) in zip(axes, metrics):
            bars = ax.bar(
                range(len(df_bar)),
                df_bar[col],
                color=[color_map[s] for s in df_bar["source"]],
                edgecolor='black',
                linewidth=0.5,
            )
            ax.set_xticks(range(len(df_bar)))
            ax.set_xticklabels(df_bar["source"], rotation=30, ha='right', fontsize=9)
            ax.set_ylabel(ylabel, fontsize=11)
            ax.set_title(title, fontsize=13, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            
            # Add value labels on bars
            for bar, val in zip(bars, df_bar[col]):
                ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        plt.savefig("benchmark_comparison_bars.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Saved: benchmark_comparison_bars.png")

In [ ]:
# --- Visualization 4: Quantization Impact (if multiple local precisions) ---

if not df_compare.empty:
    df_local_only = df_compare[df_compare["backend"] != "nim_cloud"].copy()
    
    if len(df_local_only["source"].unique()) > 1:
        df_local_single = df_local_only[df_local_only["concurrency"] == 1].copy()
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Grouped bar: TTFT by precision for each input length
        precisions = df_local_single["source"].unique()
        input_lens = sorted(df_local_single["input_length"].unique())
        
        x = np.arange(len(input_lens))
        width = 0.8 / len(precisions)
        
        for i, prec in enumerate(precisions):
            vals = []
            for il in input_lens:
                row = df_local_single[
                    (df_local_single["source"] == prec) & 
                    (df_local_single["input_length"] == il)
                ]
                vals.append(row["tokens_per_sec"].values[0] if len(row) > 0 else 0)
            
            ax.bar(x + i * width, vals, width, label=prec, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel("Input Length (tokens)", fontsize=12)
        ax.set_ylabel("Tokens/sec", fontsize=12)
        ax.set_title("Quantization Impact: Decode Throughput by Precision", fontsize=14, fontweight='bold')
        ax.set_xticks(x + width * (len(precisions) - 1) / 2)
        ax.set_xticklabels([str(il) for il in input_lens])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Calculate improvement percentages
        if len(precisions) >= 2:
            baseline = precisions[0]
            for prec in precisions[1:]:
                base_tps = df_local_single[df_local_single["source"] == baseline]["tokens_per_sec"].mean()
                prec_tps = df_local_single[df_local_single["source"] == prec]["tokens_per_sec"].mean()
                if base_tps > 0:
                    improvement = ((prec_tps - base_tps) / base_tps) * 100
                    direction = "improvement" if improvement > 0 else "regression"
                    print(f"  📊 {prec} vs {baseline}: {abs(improvement):.1f}% {direction}")
        
        plt.tight_layout()
        plt.savefig("benchmark_quantization_impact.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Saved: benchmark_quantization_impact.png")
    else:
        print("ℹ️  Only one local precision available. Quantization comparison requires multiple.")

In [ ]:
# --- Summary Table ---

if not df_compare.empty:
    print(f"\n{'=' * 80}")
    print("📊 COMPLETE BENCHMARK COMPARISON TABLE")
    print(f"{'=' * 80}")
    
    # Key metrics table
    summary_cols = ["source", "precision", "input_length", "concurrency",
                    "ttft_ms", "itl_ms", "tokens_per_sec", "total_latency_ms"]
    available_cols = [c for c in summary_cols if c in df_compare.columns]
    
    display(df_compare[available_cols].round(2))
    
    # Best results per metric
    print(f"\n🏆 Best Results:")
    df_single_512 = df_compare[(df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)]
    if not df_single_512.empty:
        best_ttft = df_single_512.loc[df_single_512["ttft_ms"].idxmin()]
        best_tps = df_single_512.loc[df_single_512["tokens_per_sec"].idxmax()]
        best_itl = df_single_512.loc[df_single_512["itl_ms"].idxmin()]
        
        print(f"  Lowest TTFT:    {best_ttft['source']} — {best_ttft['ttft_ms']:.1f} ms")
        print(f"  Highest T/s:    {best_tps['source']} — {best_tps['tokens_per_sec']:.1f} tok/s")
        print(f"  Lowest ITL:     {best_itl['source']} — {best_itl['itl_ms']:.1f} ms")

## Cell 7: Generate Report

This cell produces:
1. A complete markdown summary of all results
2. Saved chart images
3. Copy-paste ready data for blog posts or presentations

In [ ]:
# ============================================================
# Cell 7: Generate Report
# ============================================================
from datetime import datetime

report_date = datetime.now().strftime("%Y-%m-%d %H:%M")

# --- Build Report ---
report_lines = []
report_lines.append("# 🔥 Inference Engineering Benchmark Report")
report_lines.append("")
report_lines.append(f"**Date:** {report_date}")
report_lines.append(f"**GPU:** {gpu_name}")
report_lines.append(f"**VRAM:** {gpu_vram_mb/1024:.1f} GB")
report_lines.append(f"**Compute Capability:** SM {cc_major}.{cc_minor}")
report_lines.append(f"**Primary Model:** {model_id}")
report_lines.append("")

# --- Environment Summary ---
report_lines.append("## Environment")
report_lines.append("")
report_lines.append(f"| Component | Details |")
report_lines.append(f"|-----------|---------|")
report_lines.append(f"| GPU | {gpu_name} ({gpu_vram_mb/1024:.1f} GB VRAM) |")
report_lines.append(f"| Compute Capability | SM {cc_major}.{cc_minor} |")
report_lines.append(f"| TRT-LLM Available | {'✅ Yes' if trt_llm_available else '❌ No (HF Transformers fallback)'} |")
report_lines.append(f"| FP8 Native Support | {'✅ Yes' if SUPPORTS_FP8 else '❌ No (requires SM 8.9+)'} |")
report_lines.append(f"| Model | {model_id} ({model_cfg['nparams']} params) |")
report_lines.append("")

# --- Engine Build Results ---
if engine_build_results:
    report_lines.append("## TensorRT-LLM Engine Build Results")
    report_lines.append("")
    report_lines.append("| Engine | Status | Build Time | Engine Size |")
    report_lines.append("|--------|--------|------------|-------------|")
    for r in engine_build_results:
        bt = f"{r['build_time_s']:.1f}s" if r.get('build_time_s') else 'N/A'
        sz = f"{r['engine_size_mb']:.0f} MB" if r.get('engine_size_mb') else 'N/A'
        report_lines.append(f"| {r['engine']} | {r['status']} | {bt} | {sz} |")
    report_lines.append("")

# --- Benchmark Results Table ---
if not df_compare.empty:
    report_lines.append("## Benchmark Results (512-token input, single request)")
    report_lines.append("")
    
    df_summary = df_compare[(df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)].copy()
    if not df_summary.empty:
        report_lines.append("| Configuration | TTFT (ms) | ITL (ms) | Tokens/sec | Total Latency (ms) |")
        report_lines.append("|---------------|-----------|----------|------------|---------------------|")
        for _, row in df_summary.iterrows():
            report_lines.append(
                f"| {row['source']} | {row['ttft_ms']:.1f} | {row['itl_ms']:.1f} | "
                f"{row['tokens_per_sec']:.1f} | {row['total_latency_ms']:.1f} |"
            )
    report_lines.append("")
    
    # Concurrency scaling
    df_conc_summary = df_compare[df_compare["input_length"] == 512].copy()
    if len(df_conc_summary["concurrency"].unique()) > 1:
        report_lines.append("## Throughput Scaling with Concurrency (512-token input)")
        report_lines.append("")
        report_lines.append("| Configuration | Concurrency | Output Tokens/sec | Total Latency (ms) |")
        report_lines.append("|---------------|-------------|-------------------|---------------------|")
        for _, row in df_conc_summary.sort_values(["source", "concurrency"]).iterrows():
            report_lines.append(
                f"| {row['source']} | {row['concurrency']} | {row['tokens_per_sec']:.1f} | "
                f"{row['total_latency_ms']:.1f} |"
            )
        report_lines.append("")

# --- Key Insights ---
report_lines.append("## Key Insights")
report_lines.append("")

insights = []

# Insight 1: Quantization benefit
if not df_compare.empty:
    local_sources = df_compare[df_compare["backend"] != "nim_cloud"]["source"].unique()
    if len(local_sources) >= 2:
        df_512_1 = df_compare[(df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)]
        baseline_tps = df_512_1[df_512_1["source"] == local_sources[0]]["tokens_per_sec"].values
        quant_tps = df_512_1[df_512_1["source"] == local_sources[-1]]["tokens_per_sec"].values
        if len(baseline_tps) > 0 and len(quant_tps) > 0 and baseline_tps[0] > 0:
            pct = ((quant_tps[0] - baseline_tps[0]) / baseline_tps[0]) * 100
            direction = "faster" if pct > 0 else "slower"
            insights.append(f"**Quantization wins:** {local_sources[-1]} is {abs(pct):.1f}% {direction} than {local_sources[0]} at 512-token input.")

# Insight 2: Local vs Cloud
if not df_compare.empty:
    local_rows = df_compare[(df_compare["backend"] != "nim_cloud") & (df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)]
    cloud_rows = df_compare[(df_compare["backend"] == "nim_cloud") & (df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)]
    if not local_rows.empty and not cloud_rows.empty:
        best_local_tps = local_rows["tokens_per_sec"].max()
        best_cloud_tps = cloud_rows["tokens_per_sec"].max()
        best_local_ttft = local_rows["ttft_ms"].min()
        best_cloud_ttft = cloud_rows["ttft_ms"].min()
        insights.append(f"**Local vs Cloud:** Best local TTFT is {best_local_ttft:.1f}ms, cloud TTFT is {best_cloud_ttft:.1f}ms. Cloud adds network latency but runs on H100 hardware.")

# Insight 3: T4 limitations
if not SUPPORTS_FP8:
    insights.append("**GPU limitation noted:** T4 (SM 7.5) doesn't support FP8. On A100/H100 (SM 8.0+/9.0), FP8 engines would deliver ~2x throughput over FP16.")
    insights.append("**NIM advantage:** Cloud API runs on H100 infrastructure, demonstrating full FP8 optimization that T4 cannot replicate locally.")

if insights:
    for insight in insights:
        report_lines.append(f"- {insight}")
else:
    report_lines.append("- Run the benchmark cells above to generate insights.")
report_lines.append("")

# --- Print Report ---
report_text = "\n".join(report_lines)
print(report_text)

# --- Save Report ---
with open("inference_benchmark_report.md", "w") as f:
    f.write(report_text)
print(f"\n💾 Report saved: inference_benchmark_report.md")

In [ ]:
# --- Blog-Post Ready Data ---

print("=" * 70)
print("📋 COPY-PASTE READY DATA FOR BLOG POST / PRESENTATION")
print("=" * 70)
print()

if not df_compare.empty:
    # Quick summary for slides/blog
    df_512_1 = df_compare[(df_compare["concurrency"] == 1) & (df_compare["input_length"] == 512)]
    
    print("### 512-token Input, Single Request Summary")
    print("```")
    for _, row in df_512_1.iterrows():
        print(f"{row['source']:30s} | TTFT: {row['ttft_ms']:7.1f}ms | "
              f"T/s: {row['tokens_per_sec']:6.1f} | Latency: {row['total_latency_ms']:7.1f}ms")
    print("```")
    print()
    
    # CSV export for spreadsheets
    df_compare.to_csv("inference_benchmark_results.csv", index=False)
    print("💾 Full results CSV: inference_benchmark_results.csv")
    print()
    
    # Chart files list
    chart_files = [
        "benchmark_ttft_throughput.png",
        "benchmark_concurrency.png",
        "benchmark_comparison_bars.png",
        "benchmark_quantization_impact.png",
    ]
    existing_charts = [f for f in chart_files if os.path.exists(f)]
    print(f"📊 Chart files saved: {len(existing_charts)}")
    for f in existing_charts:
        size_kb = os.path.getsize(f) / 1024
        print(f"   - {f} ({size_kb:.0f} KB)")
    
    print()
    print("✅ All artifacts ready for your NVIDIA hiring portfolio!")
    print("   Use these in: blog posts, GitHub repos, presentation slides")
else:
    print("⚠️  No results to export. Run the benchmark cells first.")

---

## 🎯 Takeaways for NVIDIA Interview

### What This Notebook Demonstrates

1. **Inference Optimization Depth**: Understanding that precision selection is GPU-dependent (FP8 needs Ada+, AWQ works everywhere) shows production awareness.

2. **Full Stack Competence**: From model download → engine building → benchmarking → visualization → reporting. This is the complete inference engineering pipeline.

3. **NIM API Integration**: The OpenAI-compatible NIM endpoint is NVIDIA's flagship inference product. Using it correctly (streaming, auth, latency measurement) shows product familiarity.

4. **Statistical Rigor**: P50/P95/P99 latencies, multiple runs, warmup phases — this is how production benchmarks are done.

5. **Hardware Awareness**: The T4 limitation handling (graceful fallback, explaining why FP8 doesn't work) demonstrates deeper understanding than just running benchmarks blindly.

### Key Talking Points

- "I benchmarked TensorRT-LLM with multiple quantization formats and identified the optimal precision for each GPU generation."
- "I compared local T4 inference against NIM cloud API running on H100 infrastructure, quantifying the network overhead vs hardware advantage trade-off."
- "I built a complete benchmarking framework that handles GPU capability detection, graceful degradation, and produces publication-ready visualizations."

---

*Built with ❤️ by Abishek Bangalore Muralikrishna — NVIDIA Inference Engineering Portfolio*